In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pickle
import glob

In [4]:
df = pd.DataFrame(columns=['model', 'dataset', 'method', 'args', 'iter', 'acc@1', 'acc@5'])

paths = glob.glob('./output/*/*/*')
for path in sorted(paths):
    if not os.path.isdir(path):
        continue

    model, dataset, method = path.split('/')[-3:]
    args = ', '.join(method.split('_')[1:])
    method = method.split('_')[0]
    acc1 = []
    acc5 = []

    for iter in os.listdir(path):
        if not os.path.exists(os.path.join(path, iter, 'results.pkl')):
            continue
        
        results = pickle.load(open(os.path.join(path, iter, 'results.pkl'), 'rb'))

        acc1.append(results['accuracy1'])
        acc5.append(results['accuracy5'])
        
    if len(acc1) > 0:
        df = df.append({
            'model': model,
            'dataset': dataset,
            'method': method,
            'args': args,
            'iter': len(acc1),
            'acc@1': f'{np.mean(acc1):.2f}±{np.std(acc1):.2f}',
            'acc@5': f'{np.mean(acc5):.2f}±{np.std(acc5):.2f}'
        }, ignore_index=True)
        print(path)
df

./output/conv2next-base/cifar10/baseline_4e-3
./output/conv2next-base/cifar100/baseline_4e-3
./output/conv2next-base/stl10/baseline_4e-3
./output/conv2next-tiny/cifar10/baseline_4e-3
./output/conv2next-tiny/cifar10/snr_0.01
./output/conv2next-tiny/cifar10/snvr_0.01_1.0
./output/conv2next-tiny/cifar100/baseline_4e-3
./output/conv2next-tiny/cifar100/sn
./output/conv2next-tiny/cifar100/snr_0.01
./output/conv2next-tiny/cifar100/snvr_0.01_1.0
./output/conv2next-tiny/cifar100/wn
./output/conv2next-tiny/food101/baseline_4e-3
./output/conv2next-tiny/stl10/baseline_4e-3
./output/conv2next-tiny/stl10/snr_0.01
./output/conv2next-tiny/stl10/snvr_0.01_1.0
./output/conv2next-tiny/tiny-imagenet/baseline_4e-3
./output/conv2next-tiny/tiny-imagenet/sn
./output/conv2next-tiny/tiny-imagenet/snr_0.01
./output/conv2next-tiny/tiny-imagenet/snvr_0.01_1.0
./output/conv2next-tiny/tiny-imagenet/snvr_unified_0.01_1.0
./output/conv2next-tiny/tiny-imagenet/wn


,model,dataset,method,args,iter,acc@1,acc@5
0,conv2next-base,cifar10,baseline,4e-3,1,97.85±0.00,99.88±0.00
1,conv2next-base,cifar100,baseline,4e-3,1,83.87±0.00,95.80±0.00
2,conv2next-base,stl10,baseline,4e-3,1,91.31±0.00,99.53±0.00
3,conv2next-tiny,cifar10,baseline,4e-3,5,97.43±0.12,99.91±0.03
4,conv2next-tiny,cifar10,snr,0.01,5,97.43±0.10,99.89±0.03
5,conv2next-tiny,cifar10,snvr,"0.01, 1.0",5,97.39±0.03,99.88±0.03
6,conv2next-tiny,cifar100,baseline,4e-3,5,83.05±0.32,95.92±0.14
7,conv2next-tiny,cifar100,sn,,5,80.73±0.44,95.02±0.19
8,conv2next-tiny,cifar100,snr,0.01,5,83.36±0.15,95.86±0.13
9,conv2next-tiny,cifar100,snvr,"0.01, 1.0",5,83.48±0.22,96.05±0.16


In [ ]:
from models.conv2next import conv2next_tiny
from optimizer.snvr import calc_sn_uv
import matplotlib.pyplot as plt

paths = [
    './output/conv2next-tiny/cifar100/baseline_4e-3',
    './output/conv2next-tiny/cifar100/wn',
    './output/conv2next-tiny/cifar100/sn',
    './output/conv2next-tiny/cifar100/sn', # alter tb
    './output/conv2next-tiny/cifar100/snr_0.01',
    './output/conv2next-tiny/cifar100/snvr_0.01_1.0',
]

model = conv2next_tiny(num_classes=100)

for path in paths:
    results = pickle.load(open(os.path.join(path, iter, 'results.pkl'), 'rb'))
    ckpt = torch.load(os.path.join(path, iter, 'ckpt_best.pt'))
    print(ckpt.keys())
    model.load_state_dict(ckpt['model'])
    
    svds = []
    for layer in model.modules():
        if isinstance(layer, torch.nn.Conv2d) or isinstance(layer, torch.nn.Linear):
            weight = layer.weight.data.clone()
            sn, uv = calc_sn_uv(weight, n_power_iterations=5)
            svds.append(svd)
                 
    
    plt.figure(figsize=(7, 2))
    
    
    break


dict_keys(['model', 'optimizer', 'lr_scheduler', 'accuracy1', 'accuracy5', 'scaler', 'epoch', 'config'])
torch.Size([96, 3, 1, 1])
torch.Size([1, 96, 3])
torch.Size([24, 48, 1, 1])
torch.Size([1, 24, 48])
torch.Size([48, 24, 1, 1])
torch.Size([1, 48, 24])
torch.Size([48, 24, 1, 1])
torch.Size([1, 48, 24])
torch.Size([1, 2, 3, 3])
torch.Size([9, 1, 2])
torch.Size([192, 96, 2, 2])
torch.Size([4, 192, 96])
torch.Size([48, 96, 1, 1])
torch.Size([1, 48, 96])
torch.Size([96, 48, 1, 1])
torch.Size([1, 96, 48])
torch.Size([96, 48, 1, 1])
torch.Size([1, 96, 48])
torch.Size([1, 2, 3, 3])
torch.Size([9, 1, 2])
torch.Size([96, 1, 3, 3])
torch.Size([9, 96, 1])
torch.Size([384, 96])
torch.Size([384, 96])
torch.Size([96, 384])
torch.Size([96, 384])
torch.Size([96, 1, 3, 3])
torch.Size([9, 96, 1])
torch.Size([96, 1, 3, 3])
torch.Size([9, 96, 1])
torch.Size([384, 96])
torch.Size([384, 96])
torch.Size([96, 384])
torch.Size([96, 384])
torch.Size([96, 1, 3, 3])
torch.Size([9, 96, 1])
torch.Size([96, 1, 3,

<Figure size 700x200 with 0 Axes>